# Spoken Language Processing - Instituto Superior Técnico
### Laboratory Assignment 2 - Automatic Age Estimation Challenge
<!--[image](imgs/lab2_slp_banner.png)-->
<img src="imgs/lab2_slp_banner.png" alt="drawing" width="400"/>

# WEEK 1 - Classical models based on conventional knowledge-based features

This notebook contains the guide and code cells (some of them partially incomplete) that permit implementing two baseline systems for gender classification based on:
* frame based features and a simple generative model. 
* segment based features and a simple discriminative model.

Besides, the notebook will show how to obtain predictions and  score the systems on the development set. 

By the end of this week, students must adapt the last baseline to perform age regression.

**Read carefully the Markdown information, but also the comments inside the code cells (they provide useful information and hints), and also the code itself. The better you understand it, the easier will be modyfing it.**

## Before starting

Like in the introduction Notebook, we'll import the dummy Exception class:

In [1]:
from pf_tools import CheckThisCell

And  you need to mount Google drive if you are working on Google Colab. If you are not using Google Colab, skip or delete the following code cell:

And set-up your working directory:

In [2]:
import os

CWD = os.getcwd() 
DATADIR = os.path.abspath(os.path.join(CWD, '..', 'data'))

# Create the data directory if it doesn't exist
if not os.path.isdir(DATADIR):
    os.makedirs(DATADIR)
    print(f"Created directory: {DATADIR}")

print(f'Current working directory is set to: {CWD}')   
print(f'Your LAB2 data folder is: {DATADIR}')

Current working directory is set to: /home/luispma/slp_lab2/src
Your LAB2 data folder is: /home/luispma/slp_lab2/data


## 1. The MFCC and GMM baseline system 
This baseline consists of MFCC feature extraction  (based on the `librosa` module) with the following (optional) additional commponents:
- Delta and Double-delta computation;
- Voice Activity Detection (VAD);
- Cepstral mean and variance normalization (CMVN).

The feature extraction module is followed by GMMs of 64 dimensions for each gender class (using the `sklearn` module). 

Student groups will be graded depending on their ability to complete the different modules, propose alternatives, and evaluate and compare different configurations (delta vs delta-delta, GMM dimension, using or not using VAD or CMVN, etc.)

### 1.1 Initialization and importing modules

In [3]:
from pf_tools import SLPdata
import librosa
import numpy as np
import torch
from sklearn.mixture import GaussianMixture
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt

import time
import random
from pathlib import Path
import pickle
import opensmile

GLOBAL_SEED = 35731

np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
random.seed(GLOBAL_SEED)

GENDER_CLASSES = ('F',  'M')  
GEN2ID = {'F':0, 'M':1}
ID2GEN = dict((GEN2ID[k],k)for k in GEN2ID)

### 1.2 The MFCC feature extraction module
The next function extracts MFCCs, but there are plenty of things that can be improved. You are free to change anything you want, including the number of formal parameters. 

In [ ]:
# Read carefully this function and understand it
raise CheckThisCell ## <---- Remove this after completing/checking this cell


def mfcc_extract(filename, mono=True, duration=None, n_mfcc = 13, remove_c0=False, delta_order=0, apply_vad=False, apply_cmvn=False, skip_frames=2):
    
    sr=16000
    n_mels = 40
    n_fft = 512 
    hop_length = 160
    fmin = 50
    fmax = 7800
    

    # Load audio wav into numpy array and resample it to 16kHz
    y, _ = librosa.load(filename, sr=sr, mono=mono, duration=duration)
    
    ## OPTIONAL ADDIDITIONAL STAGES 
    # 0 - PREPROCESSING - Typical preprocessing may include normalization of audio (mean removal), 
    #                       but also speech enhancement and others more complex. 
    #                       You can try this at a later stage, this is 100% optional

    # Extract MFFCs
    # (CAUTION: Notice the transpose. Our features should have dimension (TxD))
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_fft=n_fft, 
                                n_mfcc=n_mfcc, n_mels=n_mels, 
                                hop_length=hop_length, 
                                fmin=fmin, fmax=fmax, htk=False).T

    
    ## ADDIDITIONAL STAGES - LAB WORK

    # 1 Compute deltas --> Hint you can use librosa (see below)
    if delta_order > 0:
        mfcc = compute_delta(mfcc, delta_order=delta_order)

    
    # 2 COMPUTE VAD --> Hint: You can use any vad (theshold on rms energy, something avaialble in the net, ...).
    #                         Coeff0 of MFCC is highly related with Energy and it can be used as a proxy
    #                         ATTENTION: Using a VAD may have a significant impact  
    if apply_vad:
        mfcc, _ = compute_vad(mfcc)
             
    
    # 3 APPLY CMVN --> ATTENTION: Using normalization may have a significant impact
    if apply_cmvn:
        mfcc = compute_cmvn(mfcc)
        
    # 4 Remove C0 --> In typical MFFC configuration  c0 is replaced by logEnergy, probably low/no impact
    if remove_c0:
        pass
        
    if skip_frames > 0: # remove begining and ending frames, which are typically unreliable
        mfcc = mfcc[skip_frames:-skip_frames]

    return mfcc, y


Before implementing the additional modules, let's extract MFCCs of one audio file using `mfcc_extract`. Notice that this function returns two arguments: an array containing the features and the audio waveform.

In [ ]:
audio_file = f'{DATADIR}/train_small/wav/00834c0e904d40eda496e55010acebc5.wav'
mfcc, audio = mfcc_extract(audio_file, n_mfcc=7, apply_cmvn=False)

# We can plot the audio waveform and the features
fig, ax = plt.subplots(nrows=2, sharex=False, figsize=(8, 6))
librosa.display.waveshow(audio, sr=16000,  ax=ax[0])
ax[0].set(title='Audio waveform')

# notice the transposition before calling the display: librosa expects arrays of DxT, in which T is time, 
# however our feature extraction returns arrays of TxD
librosa.display.specshow(mfcc.T,  ax=ax[1])
# fig.colorbar(img, ax=[ax[1]])
ax[1].set(title='MFCC')


Inspect the output arrays and confirm they have the expected dimension (use `audio.shape` and `mfcc.shape`). **IMPORTANT:** You must understand what is the relation between the dimensions of audio and mfcc array!?

In [ ]:
raise CheckThisCell ## <---- Remove this after completing/checking this cell
audio.shape, mfcc.shape

Now, to run the data processing stage for the `train_small` partition we will simply instanciate the SLPdata class as described in the introduction Notebook. Take a sit because it can take a bit:

In [ ]:
# We will use a dictionary to store the different feature extraction configurations
transform = {'mfcc7' : 
                 {  # audio_transform is exptected to be a function that takes as input a 
                    # filename and returns the features. For this reason, we use a lambda function
                    # to pass the parameters to the mfcc_extract function
                     'audio_transform': lambda x : mfcc_extract(x, 
                                                                mono=True, 
                                                                n_mfcc = 7, 
                                                                apply_vad=False, 
                                                                apply_cmvn=False)[0],
                     'chunk_transform': None, 
                     'chunk_size': -1, # <--- a negative value mean the whole file: we dont chunk the audio and
                                       #      we compute a feature matrix for each complete audio file
                     'chunk_hop': -1
                 }
            }


trainset = 'train_small'
transform_id = 'mfcc7'

train_small_slp = SLPdata(DATADIR, trainset, 
                 transform_id=transform_id, 
                 audio_transform=transform[transform_id]['audio_transform'], 
                 chunk_transform=transform[transform_id]['chunk_transform'],
                 chunk_size=transform[transform_id]['chunk_size'], 
                 chunk_hop=transform[transform_id]['chunk_hop']
                )


Check your current folder, many things happened!! 

Notice that if you instanciate again the SLPdata class for the 'train_small' partition, the data will not be downloaded again. 
Additionally, if there is already a folder with the name `transform_id`, feature extraction will not run again. You need to delete from your filesystem the folder with the features if you want to run again the feature extraction (using the same identifier) or , alternatively, you can change the identifier. Be careful because you can easily increase the amount of data generated. If you try a feature extraction method that provides bad results, you probably don't want to keep the features in disk.

Now that we know the basics, it is time to try to improve your features. To do this, try to define some or all of the following steps to improve your feature extraction pipeline (students will be evaluated depending on their ability to solve the following functions):

#### Feature normalization:

In [ ]:
# Numpy arrays have methods to compute mean and variance, so this one should be really easy.
# https://numpy.org/doc/stable/reference/generated/numpy.mean.html
# https://numpy.org/doc/stable/reference/generated/numpy.std.html
# 
# Be careful about the dimensions!! You want to compute mean and variance over the time dimension!!

raise CheckThisCell ## <---- Remove this after completeing/checking this cell

def compute_cmvn(features, only_mean=False):
    return features 

#### Delta computation:

In [ ]:
# librosa contains functions to compute deltas: https://librosa.org/doc/main/generated/librosa.feature.delta.html
# Ideally, this function should permit choosing 
# the maximum order (order=1 is delta, order=2 is delta-delta and so on). 
# When selecting order > 1 , it is exepected that ALL the delta components are 
# appended to the feature vector. For instance, if the features are of dimension D, 
# selecting order 2 will contactenate both the velocity and the acceleration.
# Additionally the functions must permit keeping the static MFCCs. 
# It is FOUNDAMENTAL TO KEEP THEM, DYNAMIC ALONE WILL NOT WORK WELL!!
# 
# BE CAREFUL IF YOU USE LISBROSA, it expects the time dimension to be the last one, in other words, or you  transpose
# and transpose back, or you select the proper axis to apply delta computation!!
 
raise CheckThisCell ## <---- Remove this after completeing/checking this cell

def compute_delta(features, delta_order=2, keep_static=True):
    return features



#### VAD removal:

In [ ]:
# You can think of several strattegies to compute VAD, for instance, simple ones based on 
# energy and a threshold.
# You can use librosa to obtain the energy of each frame (use same framing config as for the MFCCs)
# https://librosa.org/doc/main/generated/librosa.feature.rms.html
# Or alternatively, 
# the first mfcc coefficient, which is a good proxy for the energy
# You can also use a pretrained VAD, like SILERO: https://github.com/snakers4/silero-vad

# In addition to the features without the silence frames, 
# this function may return a sequence of booleans to help you to validate the method.
    
raise CheckThisCell ## <---- Remove this after completeing/checking this cell

def compute_vad(features, energy=None, y=None):
    vad = np.isreal(features[:,0])
    return features[vad], vad


In [ ]:
# The second returned expression of the VAD function is expected to be a boolean vector
# of one True or False, one per frame of the feature matrix. You can see the output of it and verify 
# if it's doing what is expected (removing low energy features) 
raise CheckThisCell ## <---- Remove this after completing/checking this cell

from functools import reduce

seconds = 10
mfcc_wo_vad, vad = compute_vad(mfcc,y=audio)
plt.plot(audio[:seconds*16000])
plt.plot(reduce(lambda a,b:a+b, ((0.3,)*160 if s else (0,)*160 for s in vad[:seconds*100])), 'r')


#### Test your functions

You can keep testing using an audio file with different configurations of the feature extraction Inspect the dimensions, verify that your code is doing what is expected, inspect and visualize the data using the previous examples and some of the lessons learnt in LAB1. You can also listen to some examples.

In [ ]:
raise CheckThisCell ## <---- Remove this after completeing/checking this cell

audio_file = f'{DATADIR}/train_small/wav/00834c0e904d40eda496e55010acebc5.wav'
mfcc_dd, _ = mfcc_extract(audio_file, apply_cmvn=False, delta_order=2)
mfcc_d, _ = mfcc_extract(audio_file, apply_cmvn=False, delta_order=1)
mfcc, _ = mfcc_extract(audio_file, apply_cmvn=False)


In [ ]:
import IPython.display as ipd
ipd.Audio(audio_file)

#### Run new feature extraction configurations

Once you are done completing the additional modules, you can rerun the feature extraction process for the training set using new configurations. You can add new entries to the dictionary keeping several transformation configurations and instantiate the SLPdata class:

In [ ]:
raise CheckThisCell ## <---- Remove this after completeing/checking this cell


## <--- YOU CAN ADD NEW CONFIGURATIONS
transform['mfcc13'] = {  
                     'audio_transform': lambda x : mfcc_extract(x, mono=True, n_mfcc = 7, apply_vad=False, apply_cmvn=False)[0],
                     'chunk_transform': None, 
                     'chunk_size': -1, # <--- a negative value mean the whole file: we dont chunk the audio and
                                       #      we compute a feature matrix for each complete audio file
                     'chunk_hop': -1
                 }

transform ['mfcc39_d_dd_vad_cmvn'] =  { 
                     'audio_transform': lambda x : mfcc_extract(x, mono=True, n_mfcc = 13, delta_order=2, apply_vad=True, apply_cmvn=True)[0],
                     'chunk_transform': None, 
                     'chunk_size': -1, # <--- a negative value means the whole file: we dont chunk the audio and
                                       #      we compute a feature vector for each complete audio file
                     'chunk_hop': -1
                 }
            

trainset = 'train_small'
transform_id = 'mfcc7'
# transform_id = 'mfcc39_d_dd_vad_cmvn' # <--- select the configuration you want to use

train_small_slp = SLPdata(DATADIR, trainset, 
                 transform_id=transform_id, 
                 audio_transform=transform[transform_id]['audio_transform'], 
                 chunk_transform=transform[transform_id]['chunk_transform'],
                 chunk_size=transform[transform_id]['chunk_size'], 
                 chunk_hop=transform[transform_id]['chunk_hop']
                )


### 1.3 GMMs for Gender modeling
The model in this baseline is extremely simple: we'll train an individual GMM model for each gender class on top of the features that we just extracted. Later, in prediction time, given a test audio sample, we'll compute the loglikelihood obtained with each GMM model and select as the predicted gender the one whose model gives the highest likelihood. Let's go for it!!

Notice that each training sample contains more than one frame (N dimension), so for GMM training we will first concatenate all data to have all training data in one array and the corresponding label (with same length):


In [ ]:
from pf_tools import prepare_slp_data

# IN GMM training each training sample contains more than one frame;
# so we cocatenate all data to have all training data in one array 
# and the corresponding label (with same length)

# STEP 1: Instantiate the SLPdata object with the desired configuration
#         We will train using the following configuration
transform_id = 'mfcc7' # <--- CHANGE THIS
trainset = 'train_small'   # <--- CHANGE THIS IF YOU WANT TO USE THE COMPLETE TRAINING SET
train_small_slp = SLPdata(DATADIR, trainset, 
                 transform_id=transform_id, 
                 audio_transform=transform[transform_id]['audio_transform'], 
                 chunk_transform=transform[transform_id]['chunk_transform'],
                 chunk_size=transform[transform_id]['chunk_size'], 
                 chunk_hop=transform[transform_id]['chunk_hop']
                )

# STEP 2: Concatenate all data and labels in one data structure with 'data', 'label' 
#         and 'identifer' keys.
#         Each row corresponds to a frame of the feature matrix
train_data_and_labels = prepare_slp_data(train_small_slp)
train_data, train_labels = train_data_and_labels['data'], train_data_and_labels['label']

Now we have three data structures containing the complete training dataset and the corresponding reference labels (we will not need the file indentifiers for trianing). Check the sizes. You can have a look to the content of one time instant. Notice that labels contain both gender and age. Do some checks on the data to be sure that everything is as expected:

In [ ]:
# Check the traininig data. Notice that if you apply VAD, the size of the training data must be smaller than the complete data set. 
# Register the size in frames and in time of training data for each gender

raise CheckThisCell ## <---- Remove this after completeing/checking this cell
train_data.shape, train_labels.shape

Let's go training. Again, depending of the amount of data used, the model complexity and computational resources of the machine that you're using, this can take a while. So, relax while the computer works for you!

In [ ]:
## STEP 3: TRAIN GMM models (ML) 
start = time.time()

models = {}
n_gauss = 16 ### <---- LAB WORK: You can play with the amount of Gaussians and register the impact on performance
max_iter = 50

for gender in GENDER_CLASSES:
    models[gender] = GaussianMixture(n_components=n_gauss, 
                                     covariance_type='diag', 
                                     max_iter=max_iter, 
                                     n_init=1, 
                                     init_params='kmeans', 
                                     verbose=2, 
                                     verbose_interval=1)
    
for gender in GENDER_CLASSES:
    print(f'Training model for {gender}')
    gender_label_pos = 0
    models[gender].fit(train_data[train_labels[:,gender_label_pos] == gender])
    
print(f'Finished training all GMM models in {time.time() - start}')


Once the models have been trained, we can store them in disk for later usage. Again, be careful and avoid storing versions of useless models. By default, the model is stored in a folder inside the data partition folder and contains the feature extraction in the name and the date.

In [ ]:
# STEP 4: Save the models
from pf_tools import save_model
model_id = save_model(models, f'gmm_{transform_id}', f'{DATADIR}/{trainset}/models/gender/')
print(f'Model {model_id} saved in {DATADIR}/{trainset}/models/gender/')

You can also check the `sklearn` documentation and inspect the models trained:

In [ ]:
models['M'].means_.shape

If you later need to reload your models (because you want to use them to predict on new data), you will have to do the following:

In [ ]:
raise CheckThisCell ## <---- You can skip this if you are not interested in loading the models
from pf_tools import load_model

# model_id = 'gmm_mfcc7_2026-05-06_15:54:37' # <--- CHANGE THIS ACCORDING TO THE MODEL YOU WANT TO LOAD

filename = f'{DATADIR}/{trainset}/models/gender/{model_id}/model.pkl'
models = load_model(filename)


### 1.4 Classification of the dev set

Now that we  already have trained models, let's predict/identify the speakers gender in new audio data and test our model!!! 

But first, we need to obtain the development partition and apply the same feature extraction as previously (using the SLPdata class).

**IMPORTANT WARNING** Make sure to use the exact same feature extraction configuration as the one used for the train set. Otherwise, your model will be in disagreement with your evaluation data, and very likely, will not work at all.


In [ ]:

raise CheckThisCell ## <---- Remove this after completeing/checking this cell

transform_id = 'mfcc7' ## <--- CHANGE THIS?

dev_slp = SLPdata(DATADIR,'dev', 
                 transform_id=transform_id, 
                 audio_transform=transform[transform_id]['audio_transform'], 
                 chunk_transform=transform[transform_id]['chunk_transform'],
                 chunk_size=transform[transform_id]['chunk_size'], 
                 chunk_hop=transform[transform_id]['chunk_hop']
                )

Now, we will use the SLPdata instance to load the data and use the models for scoring. 

First, we will store the data in a dictionary, with keys corresponding to each file of the development and with value a dictionary containing the 'data' and 'label'. We will use the previous `prepare_slp_data` function with different options:

In [ ]:
dev_data = prepare_slp_data(dev_slp, collapse_samples=False, expand_labels=False)

Now, we will compute the log-likelihood for every file and gender model, obtain the prediction and store everything (including the filenames and the reference label) in a dictionary. We provide a gmm_predict function to do this:

In [ ]:
from pf_tools import gmm_predict

start = time.time()
gender_label_pos = 0
results_dev = gmm_predict(models, dev_data, GENDER_CLASSES, label_pos=gender_label_pos)
print(f'Finished predicting all data in {time.time() - start}')


You can have a look to the contents of this results structure:

In [ ]:
results_dev['hyp'][:10], results_dev['ref'][:10], results_dev['fileids'][:10], results_dev['llhs'][:10,:]


We will save the results object of the development set. We will use it later to generate the final submission file:

In [ ]:
# save results structure 
# model_id = 'gmm_mfcc7_2026-05-06_15:54:37' # <--- CHANGE THIS ACCORDING TO THE MODEL YOU WANT TO LOAD
filename = f'{DATADIR}/{trainset}/models/gender/{model_id}/dev.pkl'
pickle.dump(results_dev, open(filename, 'wb'))


### 1.5 System Evaluation
After running the previous cells, we obtained two arrays with the reference and hypothesis labels (we can also reload them in case we need them). We can use these to compute different evaluation metrics and inspect the performance (and potential problems) of our system. Of course, you will only be able to do this assessment with the development set, since you don't have access to the eval labels.

You can for instance obtain a classification report summary:

In [ ]:
ref, hyp = results_dev['ref'], results_dev['hyp']
print(classification_report(ref, hyp, target_names=GENDER_CLASSES))

Overall accuracy (this will be the **main metric for system ranking**):

In [ ]:
accuracy_score(ref, hyp)

Or a confusion matrix: 

In [ ]:
cm = confusion_matrix(ref, hyp)
cm

and visualize it:

In [ ]:
from pf_tools import plot_confusion_matrix
plot_confusion_matrix(cm, GENDER_CLASSES, title=f'Confusion Matrix\n(Accuracy {100*accuracy_score(ref, hyp):.2f})')

As a form of approximate reference, these are the accuracies that can be obtained with the following configurations using the `train_small` data set:

|       |     Accuracy       |
|-------|--------------------|
| mfcc7 (16 gauss)| 94.02    |

These results are very good. Even with a very simple baseline, it is possible to achieve excellent results in gender classification. This is a pretty simple task. Age regression will be different.

### 1.6 Classification of the evl partition

Once you are happy with your system and the results obtained in the development set, you are ready to generate the predictions on the `'evl'` partition. To do that, you have to follow the same process as for the development partition, but of course, this time you will not be able to obtain performance results because you don't have labels for this partition. 

We start by instantiating the `SLPdata` class for the `'evl'` partition:


In [ ]:

raise CheckThisCell ## <---- Remove this after completeing/checking this cell

# RELOAD MODEL!?!?

# model_id = 'gmm_mfcc7_2026-05-06_15:54:37' # <--- CHANGE THIS ACCORDING TO THE MODEL YOU WANT TO LOAD
# filename = f'{DATADIR}/{trainset}/models/gender/{model_id}/model.pkl'
# models = pickle.load(open(filename, 'rb'))


evl_slp = SLPdata(DATADIR,'evl', 
                 transform_id=transform_id, 
                 audio_transform=transform[transform_id]['audio_transform'], 
                 chunk_transform=transform[transform_id]['chunk_transform'],
                 chunk_size=transform[transform_id]['chunk_size'], 
                 chunk_hop=transform[transform_id]['chunk_hop']
                )


Then, we load the evaluation data and apply the model(s) to the new `'evl'` data. We will also save the results for later use. 

In [ ]:
evl_data = prepare_slp_data(evl_slp, collapse_samples=False, expand_labels=False)
gender_label_pos = 0

start = time.time()
results_evl = gmm_predict(models, evl_data, GENDER_CLASSES, label_pos=gender_label_pos)
filename = f'{DATADIR}/{trainset}/models/gender/{model_id}/evl.pkl'
pickle.dump(results_evl, open(filename, 'wb'))

print(f'Finished predicting all data in {time.time() - start}')

## 2. The openSMILE and SVM baseline system 
This second conventional baseline consists of a segment (file) based feature extraction module. That is, for each data file, a single feature vector will be extracted. In particular, we will use the popular `openSMILE` toolkit.

The feature extraction step is followed by a simple linear SVM (using the `sklearn` module). 

Student groups will be graded depending on their ability to explore different feature configurations and model alternatives/configurations.

This set-up will be the basis for our age estimator, that  will represent each audio file as single vector followed by a SV regressor.

### 2.1 The openSMILE feature extraction module
[openSMILE](https://audeering.github.io/opensmile-python/) is a popular feature extraction toolkit, specially devoted to speech paralinguistic tasks. It is based in a two stage approach: 

1) It extracts low-level descriptors at the frame level (just like previous MFCCs)
2) It applies a set of functionals that summarize a time-varying feature sequence into a fixed-size one dimensional feature vector.

For this baseline system, we will use functional level features. Have a look to  and run the following code snipet:


In [ ]:
# https://audeering.github.io/opensmile-python/

smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.ComParE_2016,
    feature_level=opensmile.FeatureLevel.Functionals
)
audio_file = f'{DATADIR}/train_small/wav/00834c0e904d40eda496e55010acebc5.wav'
df = smile.process_file(audio_file) # <-- This is a pandas dataframe

You can explore the contents of the dataframe, convert to a numpy array and check the dimensions. Then, write the function of two arguments that receive an audio filename and an smile object and returns the numpy feature array.

In [ ]:
raise CheckThisCell
print(df.columns.to_list()) 


In [ ]:
raise CheckThisCell

# This must return a numpy array of shape (1,X). X depends on the opensmile configuration 
def extract_smile(filename, smile):
    pass ## LABWORK -- write this tiny function (just one line!)

Exactly like in the GMM baseline system, we will create a configuration transformation and instantiate the SLPdata class to do the feature extraction for each data partition. We will store all the SLPdata objects in a dictionary.

In [ ]:
# transform = {}

transform['compare2016'] = {
    'audio_transform': 
        lambda x : extract_smile(x, 
            smile = opensmile.Smile(
                                feature_set=opensmile.FeatureSet.ComParE_2016, # <-- Change this to the feature set you want to use
                                feature_level=opensmile.FeatureLevel.Functionals,
                            )
            ), 
    'chunk_transform': None,
    'chunk_size': 0,
    'chunk_hop':0   
}

transform['gemaps'] = {
    'audio_transform': 
        lambda x : extract_smile(x, 
            smile = opensmile.Smile(
                                feature_set=opensmile.FeatureSet.GeMAPSv01b, # <-- Change this to the feature set you want to use
                                feature_level=opensmile.FeatureLevel.Functionals,
                            )
            ), 
    'chunk_transform': None,
    'chunk_size': 0,
    'chunk_hop':0   
}

In [ ]:

transform_id = 'gemaps' # <--- Change this to the configuration you want to use

slp_partitions = {}
# for partition in ('train', 'train_small', 'dev', 'evl'):
for partition in ('train_small', 'dev', 'evl'):
    slp_partitions[partition] = SLPdata(DATADIR, partition, 
                    transform_id=transform_id, 
                    audio_transform=transform[transform_id]['audio_transform'], 
                    chunk_transform=transform[transform_id]['chunk_transform'],
                    chunk_size=transform[transform_id]['chunk_size'], 
                    chunk_hop=transform[transform_id]['chunk_hop']
                    )


And we will load the data and labels of all paritions to be get prepared for training and evaluation.

In [ ]:
from pf_tools import prepare_slp_data

#   Concatenate all data and labels
#   Each row corresponds to a file
#   We store the data, labels and file identifiers of each partition in dictionaries 
#     with the partition name as key


gender_label_pos = 0
age_label_pos = 1

data, labels_gender, labels_age, fileids = {}, {}, {}, {}
# for partition in ('train', 'train_small', 'dev', 'evl'):
for partition in ('train_small', 'dev', 'evl'):
    data_and_labels = prepare_slp_data(slp_partitions[partition])
    print(f'Partition: {partition}')
    print(f'Number of samples: {data_and_labels["data"].shape[0]}')
    print(f'Number of features: {data_and_labels["data"].shape[1]}')
    print(f'Number of labels: {len(np.unique(data_and_labels["label"][:,gender_label_pos]))}')
    print(f'Number of identifiers (samples): {len(np.unique(data_and_labels["identifiers"]))}')
    print('---')
    data[partition] = data_and_labels['data']
    labels_gender[partition] = data_and_labels['label'][:,gender_label_pos]
    labels_age[partition] = data_and_labels['label'][:,age_label_pos]
    if partition != 'evl':
        labels_age[partition] = np.float32(labels_age[partition])
    fileids[partition] = data_and_labels['identifiers']
    
    

### 2.2 SVM training and evaluation
This baseline system is based on segment/file level features. This means that we have a single feature vector per file. Models like Support Vector Machines excel in taks with small training samples (and high feature dimensionality). We will use scikit learn for model training. We will use a linear SVM, but students are encouraged to try other similar models and configurations.

Have a look to the the following code snipet, it will train the model, run prediction on both dev and evl partitions and save the results:

In [ ]:
from sklearn.svm import SVC
from pf_tools import save_model

trainset = 'train_small'

# Train a linear SVM
model = SVC(kernel='linear') ### <---- a linear SVM
model.fit(data[trainset], labels_gender[trainset])  ## <---- train MODEL

model_id = save_model(model, f'svm_{transform_id}', f'{DATADIR}/{trainset}/models/gender/')
print(f'Model {model_id} saved in {DATADIR}/{trainset}/models/gender/')

# Predict the dev and evl sets
dev_results = model.predict(data['dev']) #  Predict dev
filename = f'{DATADIR}/{trainset}/models/gender/{model_id}/dev.pkl'
pickle.dump({'hyp':dev_results, 'fileids':fileids['dev']}, open(filename, 'wb'))

evl_results = model.predict(data['evl']) #  Predict evl
filename = f'{DATADIR}/{trainset}/models/gender/{model_id}/evl.pkl'
pickle.dump({'hyp':evl_results, 'fileids':fileids['evl']}, open(filename, 'wb'))


Let's see the results we obtain in the dev partition:

In [ ]:
from pf_tools import plot_confusion_matrix
ref, hyp = labels_gender['dev'], dev_results
print(classification_report(ref, hyp))
print(accuracy_score(ref, hyp))
cm = confusion_matrix(ref, hyp)
plot_confusion_matrix(cm, GENDER_CLASSES, title=f'Confusion Matrix\n(Accuracy {100*accuracy_score(ref, hyp):.2f})')


As a form of approximate reference, the linear SVM with gemapsv01 achieves an accuracy of ~99.15% using the `train_small` data set.

## 3 Develop the Age estimator baseline

Based on the openSMILE + SVM example, it is now very easy to adapt to an age regression task.

Notice that we have previously created the structures labels_gender and labels_age, containing the ground-truth for gender and age respetively. We can now use the later and change the SVM with a SVR model:

In [ ]:
from sklearn.svm import SVR
from pf_tools import save_model

trainset = 'train_small'

# Train a linear SVR for age regression
model = SVR(kernel='linear') ### <---- a linear SVR
model.fit(data[trainset], np.float32(labels_age[trainset]))  ## <---- train MODEL

model_id = save_model(model, f'svr_{transform_id}', f'{DATADIR}/{trainset}/models/age/')
print(f'Model {model_id} saved in {DATADIR}/{trainset}/models/age/')

# Predict the dev and evl sets
dev_results = model.predict(data['dev']) #  Predict dev
filename = f'{DATADIR}/{trainset}/models/age/{model_id}/dev.pkl'
pickle.dump({'hyp':dev_results, 'fileids':fileids['dev']}, open(filename, 'wb'))

evl_results = model.predict(data['evl']) #  Predict evl
filename = f'{DATADIR}/{trainset}/models/age/{model_id}/evl.pkl'
pickle.dump({'hyp':evl_results, 'fileids':fileids['evl']}, open(filename, 'wb'))

Let's see the results we obtain now in the age estimation class in the dev partition. Notice that instead of Accuracy, for the regression we will use Mean Absolute Error and Mean Square Error:

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

ref, hyp = labels_age['dev'], dev_results

print(f'Mean Absolute Error: {mean_absolute_error(ref, hyp):.2f}')
print(f'Mean Squared Error: {mean_squared_error(ref, hyp):.2f}')

The MAE result obtained with this simple system is about 9.4.

This is not very strong. Of course, the task is more difficult!!
Now you can try to play with the feature extraction and SVR configurations to improve the performance of the openSMILE + SVR age estimator. 

In the next sessions, we will explore richer speech reprentations that will contribute to obtain a better system.

## 4 Submitting your systems predictions to the challenge

The challenge will focus only on the age estimation task. 

The predictions file used for submission and scoring is a CSV file containing the predictions of both the `dev` and `evl` partitions.
The file has two fields: fileId and Age. The fileId is the unique audio file identifier and the Age field is the age estimation (real number). The predictions file name must be as follows:

`G<YY>_<SYSTEMID>.csv` 

where `<YY>` is the students' group number (use 2 digits) and `<SYSTEMID>` is an identifying string for that submission/system.
You can use the `create_submission_function` to create the csv file as follows: 

In [ ]:
from pf_tools import create_submission_file

students_group = '00' # <--- CHANGE THIS ACCORDINGLY

age_model_id = 'svr_gemaps_2026-05-06_17:12:33'
submission_name = 'BASELINE1'

age_path_results = f'{DATADIR}/{trainset}/models/age/{age_model_id}/'
csvfile = f'{CWD}/g{students_group}_{trainset}_{submission_name}.csv' # <--- CHANGE THIS ACCORDINGLY

create_submission_file(age_path_results, csvfile)


Finally, you can submit your prediction(s) in the following [Kaggle competition](https://www.kaggle.com/competitions/ist-slp-26-age-estimation-challenge) .


# Contacts and support
You can contact the professors during the classes or the office hours.

Particularly, for this second laboratory assignment, you should contact Prof. Alberto Abad: alberto.abad@tecnico.ulisboa.pt



